# Quasi-Gaussian Model Analysis

In [ ]:
using Pkg
Pkg.activate("../.")

In [ ]:
using CSV
using DataFrames
using DiffFusion
using Plots
using Printf
using StatsBase

gr()

In [ ]:
include("volatilities.jl")
include("plots.jl")

In [ ]:
file_name = "swaption_2.csv"
table = CSV.read(file_name, DataFrame)

In [ ]:
plot_smiles(table, [1.0, 2.0, 5.0, 10.0], [1.0, 2.0, 5.0, 10.0], show_title = false)

In [ ]:
table_ = table[
    (table.exercise_time .== 2.0) .&
    (table.swap_time .== 5.0), :
]

atm_volatilities = table_[:,"0"][1]

## Quasi-Gaussian Model

In [ ]:
ch = DiffFusion.correlation_holder("Std")
DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_2", 0.8)
DiffFusion.set_correlation!(ch, "EUR_f_2", "EUR_f_3", 0.8)
DiffFusion.set_correlation!(ch, "EUR_f_1", "EUR_f_3", 0.5)
#
delta = DiffFusion.flat_parameter([ 1., 7., 15. ])
chi = DiffFusion.flat_parameter([ 0.01, 0.10, 0.30 ])
times =  [ 1., 2., 5., 10. ]
values = [ 80. 70. 60. 50.;
           80. 70. 60. 50.;
           80. 70. 60. 50.] * 1.0e-4
sigma_f = DiffFusion.backward_flat_volatility("EUR", times, values)
quanto_model = nothing

gaussian_model = DiffFusion.gaussian_hjm_model("Std", delta, chi, sigma_f, ch, nothing)

#
slope_d_vals = [
    1. 1. 1. 1.;
    1. 1. 1. 1.;
    1. 1. 1. 1.
] * 1.0e-2 * (2.0)
#
slope_u_vals = [
    1. 1. 1. 1.;
    1. 1. 1. 1.;
    1. 1. 1. 1.
] * 1.0e-2 * (3.0)
#
sigma_min = 1.0e-4
sigma_max = 500. * 1.0e-4
#
ou_model = DiffFusion.ornstein_uhlenbeck_model(
    "OU",
    DiffFusion.flat_parameter("Std", 0.10),  # chi
    DiffFusion.flat_volatility("Std", 1.00),  # sigma_x
)

slope_d = DiffFusion.backward_flat_parameter("Std", times, slope_d_vals)
slope_u = DiffFusion.backward_flat_parameter("Std", times, slope_u_vals)
volatility_model = nothing  # ou_model
volatility_function =  nothing  # exp
#
m0 = DiffFusion.quasi_gaussian_multi_factor_model(
    gaussian_model, slope_d, slope_u, sigma_min, sigma_max,
    volatility_model, volatility_function,
)

sim_times = simulation_times(10.0, 1.0)
n_paths = 2^13

@time sim0 = DiffFusion.simple_simulation(
    gaussian_model, ch, sim_times, n_paths, with_progress_bar = false, # brownian_increments = DiffFusion.sobol_brownian_increments
)


m = DiffFusion.diagonal_model("Std", [m0,  ou_model])

sim_times = vcat([0.0],  reverse([ 0.5^k for k in 2:8 ]), collect(0.50:0.25:10.0))
n_paths = 2^13
@time sim1 = DiffFusion.quasi_gaussian_simulation(
    m0, ch, sim_times, n_paths, with_progress_bar = false, # brownian_increments = DiffFusion.sobol_brownian_increments
)


In [ ]:
yts = DiffFusion.flat_forward(0.03)
rel_strikes = extract_numbers(names(table)) .* 1e-4

df1 = implied_volatilities(sim1, [1.0, 2.0, 5.0, 10.0] , [2.0, 5.0, 10.0], rel_strikes, yts)

In [ ]:
plot_smiles(df1, [1.0, 2.0, 5.0, 10.0] , [2.0, 5.0, 10.0]; show_label = true, show_title = true)

In [ ]:
calib_table = table[
    in.(table.exercise_time, Ref( [2.0, 5.0, 10.0])) .&
    in.(table.swap_time, Ref([2.0, 5.0, 10.0])), :
]

In [ ]:
function atm_volatility(table, exercise_time, swap_time)
    v = table[
        (table.exercise_time .== exercise_time) .&
        (table.swap_time .== swap_time), "0"]
    return v[begin]
end


function calibrated_gaussian_model(
    model::DiffFusion.GaussianHjmModel,
    table::DataFrame,
    yts::DiffFusion.YieldTermstructure;
    max_iter::Integer = 5,
    volatility_regularisation::Float64 = 0.0,
    scaling_type::DiffFusion.BenchmarkTimesScaling = DiffFusion.ForwardRateScaling,
    swap_maturity_indices::Union{AbstractVector, Nothing} = nothing,
    swap_maturity_weights::Union{AbstractVector, Nothing} = nothing,
    )
    exercise_times = sort(unique(table.exercise_time))
    swap_times = sort(unique(table.swap_time))
    atm_volatilities = [
        atm_volatility(table, exercise_time, swap_time)
        for exercise_time in exercise_times, swap_time in swap_times
    ]
    res = DiffFusion.gaussian_hjm_model(
        model.alias,
        model.delta,
        model.chi,
        model.correlation_holder,
        exercise_times,
        swap_times,
        atm_volatilities,
        yts,
        max_iter = max_iter,
        volatility_regularisation = volatility_regularisation,
        scaling_type = scaling_type,
        swap_maturity_indices = swap_maturity_indices,
        swap_maturity_weights = swap_maturity_weights,
    )
    return res
end


res = calibrated_gaussian_model(
    gaussian_model,
    calib_table,
    yts,
    max_iter = 100,
    volatility_regularisation = 0.1,
    swap_maturity_indices = [
        [1, 2, 3],
        [1, 2, 3],
        [1, 2, 3],
    ],
    swap_maturity_weights = [
        [ 1.0, 1.0, 1.0e+1, ],
        [ 1.0, 1.0e+1, 1.0, ],
        [ 1.0e+1, 1.0, 1.0, ],
    ],
)

display(res.model.sigma_T.sigma_f.values' * 1e+4)
res.fit * 1e+4

In [ ]:
sim_times = simulation_times(10.0, 1.0)
n_paths = 2^16

@time sim = DiffFusion.simple_simulation(
    res.model, ch, sim_times, n_paths, with_progress_bar = false, # brownian_increments = DiffFusion.sobol_brownian_increments
)


In [ ]:
slope_d_vals = [
    1. 1. 1.;
    1. 1. 1.;
    1. 1. 1.
] * 1.0e-2 * (2.0)
#
slope_u_vals = [
    1. 1. 1.;
    1. 1. 1.;
    1. 1. 1.
] * 1.0e-2 * (3.0)
#
sigma_min = 1.0e-4
sigma_max = 500. * 1.0e-4
times = [ 2.0, 5.0, 10.0 ]
slope_d = DiffFusion.backward_flat_parameter("Std", times, slope_d_vals)
slope_u = DiffFusion.backward_flat_parameter("Std", times, slope_u_vals)
volatility_model = nothing
volatility_function =  nothing
#
m = DiffFusion.quasi_gaussian_multi_factor_model(
    res.model, slope_d, slope_u, sigma_min, sigma_max,
    volatility_model, volatility_function,
)

sim_times = vcat([0.0],  reverse([ 0.5^k for k in 2:8 ]), collect(0.50:0.25:10.0))
n_paths = 2^13
@time sim1 = DiffFusion.quasi_gaussian_simulation(
    m, m.gaussian_model.correlation_holder, sim_times, n_paths, with_progress_bar = false, # brownian_increments = DiffFusion.sobol_brownian_increments
)

In [ ]:
df = implied_volatilities(sim, [1.0, 2.0, 5.0, 10.0] , [1.0, 2.0, 5.0, 10.0], rel_strikes, yts)

plot_smiles(table, [1.0, 2.0, 5.0, 10.0], [1.0, 2.0, 5.0, 10.0], show_title = false, size=(1600, 1000), table_compare = df)